In [ ]:
"""
Run identity + utility + Blinder anonymization on a *custom* ECG sample
using models trained in ecg_blinder_pipeline_with_overlay.py.

Usage:
    python scripts/ecg_blinder_inference_custom.py /path/to/ecg1.mat
"""

In [ ]:
import os
import sys
import json
import argparse

In [ ]:
import numpy as np
import scipy.io as sio
from scipy.signal import resample
import torch

In [ ]:
# -------------------------------------------------------------------
# Import from your main pipeline
# -------------------------------------------------------------------
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.abspath(os.path.join(SCRIPT_DIR, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
from scripts.ecg_blinder_pipeline_with_overlay import (
    Config,
    ECGBlinderVAE,
    IdentityNet,
    build_identity_meta,
    build_utility_labels_from_superclass,
    train_or_load_utility_model_from_ptbxl,
    rmse,
    psd_correlation,
    plot_fft,
    plot_overlay_ecg,
)
from pipeline.datasets import ecg_ptbxl

-------------------------------------------------------------------
Load trained components (PTB-XL, models, scaler, PTB stats)
-------------------------------------------------------------------

In [ ]:
def load_trained_components(cfg: Config):
    """
    Reload everything needed from disk:
      - PTB-XL data (for shapes + utility)
      - PTB per-lead mean/std (for device normalization)
      - utility model + scaler
      - identity model
      - Blinder VAE
    """
    print("[Init] Loading PTB-XL data (for utility + shapes)...")
    data_dict = ecg_ptbxl.load_ptbxl_and_eda(
        ptbxl_root=cfg.datafolder,
        sampling_rate=cfg.sampling_frequency,
        output_dir=os.path.join(cfg.results_dir, "ptbxl_eda_infer"),
        save_csv=False,
    )
    X = data_dict["X"]   # (N, T, C=12)
    Y = data_dict["Y"]

    # ------------------------------------------------------------------
    # PTB-XL per-lead mean/std (for mapping device data into PTB space)
    # ------------------------------------------------------------------
    # Shape: mean/std over (N, T) for each lead C
    ptb_mean_per_lead = X.mean(axis=(0, 1))  # (C,)
    ptb_std_per_lead = X.std(axis=(0, 1))    # (C,)

    print("[Init] PTB-XL per-lead stats:")
    print("  mean:", ptb_mean_per_lead)
    print("  std: ", ptb_std_per_lead)

    # Utility labels + model + scaler + standardized PTB-XL
    print("[Init] Building PTB-XL utility labels + loading utility model...")
    y_util, super_classes = build_utility_labels_from_superclass(Y)
    util_model, util_val_metrics, data_std, scaler = train_or_load_utility_model_from_ptbxl(
        X, Y, y_util, cfg
    )

    # Identity meta + model
    print("[Init] Building identity meta + loading identity model...")
    meta, id_info = build_identity_meta(Y, cfg)
    T, C = data_std.shape[1], data_std.shape[2]

    model_id = IdentityNet(
        in_channels=C,
        n_patients=id_info["n_patients"],
        n_age_bins=id_info["n_age_bins"],
        n_height_bins=id_info["n_height_bins"],
        n_weight_bins=id_info["n_weight_bins"],
    )
    assert os.path.exists(cfg.identity_ckpt), f"Missing identity ckpt at {cfg.identity_ckpt}"
    state = torch.load(cfg.identity_ckpt, map_location=cfg.device)
    model_id.load_state_dict(state)
    model_id.to(cfg.device)
    model_id.eval()

    # Blinder VAE
    print("[Init] Loading Blinder VAE...")
    blinder_model = ECGBlinderVAE(T=T, C=C, z_dim=cfg.blinder_z_dim).to(cfg.device)
    assert os.path.exists(cfg.blinder_ckpt), f"Missing blinder ckpt at {cfg.blinder_ckpt}"
    state_blinder = torch.load(cfg.blinder_ckpt, map_location=cfg.device)
    blinder_model.load_state_dict(state_blinder)
    blinder_model.eval()

    return {
        "X_ptb": X,
        "Y_ptb": Y,
        "y_util": y_util,
        "data_std": data_std,
        "scaler": scaler,
        "meta": meta,
        "id_info": id_info,
        "util_model": util_model,
        "model_id": model_id,
        "blinder_model": blinder_model,
        "ptb_mean_per_lead": ptb_mean_per_lead,
        "ptb_std_per_lead": ptb_std_per_lead,
    }

-------------------------------------------------------------------
Custom ECG loading + alignment + PTB-shape normalization
-------------------------------------------------------------------

In [ ]:
def _extract_wlpkt_signal(mat: dict) -> np.ndarray:
    """
    Extract a 1D ECG-like signal from the WLpkt struct in your .mat file.

    Expected structure:
      WLpkt: ndarray of shape (1, N_packets)
        each element is a struct with field 'data' of shape (4, 10)

    Steps:
      - stack packets -> (N_packets, 4, 10)
      - reshape -> (4, N_packets*10)
      - average across 4 channels -> (T,)
    """
    if "WLpkt" not in mat:
        raise KeyError("Expected key 'WLpkt' in MAT file. Adjust loader if variable name differs.")

    pkt = mat["WLpkt"]

    if not (isinstance(pkt, np.ndarray) and pkt.dtype.names is not None and "data" in pkt.dtype.names):
        raise TypeError("WLpkt does not look like the expected MATLAB struct array with a 'data' field.")

    n_packets = pkt.shape[1]
    data_list = [pkt[0, i]["data"] for i in range(n_packets)]  # each (4, 10)
    data_arr = np.stack(data_list, axis=0)  # (N_packets, 4, 10)

    sig_4xT = data_arr.transpose(1, 0, 2).reshape(4, -1)  # (4, T)
    sig_1d = sig_4xT.mean(axis=0).astype(np.float32)      # (T,)

    return sig_1d

In [ ]:
def load_and_align_custom_ecg(
    mat_path: str,
    T_target: int,
    C_target: int,
    ptb_mean_per_lead: np.ndarray,
    ptb_std_per_lead: np.ndarray,
) -> np.ndarray:
    """
    Load custom ECG from .mat and make it shape (T_target, C_target),
    *mapped into PTB-XL unit/shape space*.

    Steps:
      1. Extract 1D device signal from WLpkt.
      2. Resample to T_target.
      3. Z-score (zero-mean, unit-variance) the 1D signal.
      4. For each lead c, build:
           x_c(t) = z(t) * ptb_std[c] + ptb_mean[c]
         so mean/std match PTB-XL per-lead stats, but shape is preserved.
      5. Return (T_target, C_target).
    """
    print(f"[Custom ECG] Loading {mat_path}")
    mat = sio.loadmat(mat_path)

    x = _extract_wlpkt_signal(mat)  # (L,)
    L_orig = x.shape[0]
    print(f"[Custom ECG] Raw device signal length: {L_orig} samples")
    print(f"[Custom ECG] Raw stats: mean={x.mean():.3f}, std={x.std():.3f}")

    # Resample to PTB-XL time length
    if L_orig != T_target:
        print(f"[Custom ECG] Resampling from {L_orig} -> {T_target} samples")
        x_resamp = resample(x, T_target)
    else:
        x_resamp = x

    # Z-score the device waveform (per single channel)
    x_mean = np.mean(x_resamp)
    x_std = np.std(x_resamp)
    eps = 1e-6
    x_z = (x_resamp - x_mean) / (x_std + eps)
    print(f"[Custom ECG] After z-score: mean={x_z.mean():.3f}, std={x_z.std():.3f}")

    # Map into PTB-XL per-lead mean/std space
    assert len(ptb_mean_per_lead) == C_target
    assert len(ptb_std_per_lead) == C_target

    ecg_tc = np.zeros((T_target, C_target), dtype=np.float32)
    for c in range(C_target):
        ecg_tc[:, c] = x_z * ptb_std_per_lead[c] + ptb_mean_per_lead[c]

    print("[Custom ECG] After PTB-space mapping:")
    print("  per-lead mean:", ecg_tc.mean(axis=0))
    print("  per-lead std: ", ecg_tc.std(axis=0))

    return ecg_tc

-------------------------------------------------------------------
Standardization / inverse-standardization
-------------------------------------------------------------------

In [ ]:
def standardize_ecg(ecg_tc: np.ndarray, scaler) -> np.ndarray:
    """
    Apply the same StandardScaler used for PTB-XL to the custom ECG.

    IMPORTANT: the original pipeline fits the scaler on a flattened
    (N*T*C, 1) array, i.e. one scalar "feature" = amplitude.
    So here we must mimic that and reshape to (-1, 1) before transform.

    ecg_tc: (T, C) in PTB-like units
    returns: (1, T, C) standardized
    """
    T, C = ecg_tc.shape

    # (T*C, 1) to match how the scaler was originally fitted
    flat = ecg_tc.reshape(-1, 1)
    flat_std = scaler.transform(flat)          # (T*C, 1)

    # back to (1, T, C)
    ecg_std = flat_std.reshape(1, T, C).astype(np.float32)
    return ecg_std

In [ ]:
def inverse_standardize_ecg(ecg_std_tc: np.ndarray, scaler) -> np.ndarray:
    """
    Inverse transform standardized ECG back to raw PTB-like units.

    ecg_std_tc: (1, T, C)
    returns: (T, C)
    """
    _, T, C = ecg_std_tc.shape

    # (T*C, 1) to match how scaler expects input
    flat_std = ecg_std_tc.reshape(-1, 1)
    flat_raw = scaler.inverse_transform(flat_std)   # (T*C, 1)

    ecg_raw = flat_raw.reshape(T, C).astype(np.float32)
    return ecg_raw

-------------------------------------------------------------------
Identity / utility forward passes
-------------------------------------------------------------------

In [ ]:
def run_identity_on_ecg(model_id, ecg_std_tc: np.ndarray, cfg: Config):
    """
    Run IdentityNet on a single standardized ECG.
    ecg_std_tc: (1, T, C)
    """
    x = np.transpose(ecg_std_tc, (0, 2, 1)).astype(np.float32)  # (B,C,T)
    xb = torch.from_numpy(x).to(cfg.device)

    with torch.no_grad():
        out = model_id(xb)

    preds = {
        key: out[key].argmax(dim=1).cpu().item()
        for key in ["patient", "sex", "age", "height", "weight"]
    }
    return preds

In [ ]:
def run_utility_on_ecg(util_model, ecg_std_tc: np.ndarray):
    """
    Run fastai utility model on a single standardized ECG.
    ecg_std_tc: (1, T, C)
    """
    probs = util_model.predict(ecg_std_tc)  # (1, K)
    probs = np.nan_to_num(probs, nan=0.0, posinf=5.0, neginf=-5.0)
    return probs[0]

-------------------------------------------------------------------
Anonymization for a single ECG
-------------------------------------------------------------------

In [ ]:
def anonymize_single_ecg(blinder_model, ecg_std_tc: np.ndarray, cfg: Config) -> np.ndarray:
    """
    Use Blinder VAE to anonymize a single standardized ECG.
    ecg_std_tc: (1, T, C)
    returns: (1, T, C)
    """
    blinder_model.eval()
    X = ecg_std_tc.astype(np.float32)
    with torch.no_grad():
        xb = torch.from_numpy(X).to(cfg.device)
        recon, _, _ = blinder_model(xb)
        X_anon = recon.cpu().numpy()
    X_anon = np.nan_to_num(X_anon, nan=0.0, posinf=5.0, neginf=-5.0)
    return X_anon

-------------------------------------------------------------------
Main
-------------------------------------------------------------------

In [ ]:
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("mat_path", type=str, help="Path to custom ECG .mat file")
    parser.add_argument(
        "--outdir",
        type=str,
        default=None,
        help="Output directory for FFT/overlay plots + JSON",
    )
    args = parser.parse_args()

    cfg = Config()
    if args.outdir is None:
        outdir = os.path.join(cfg.results_dir, "custom_ecg")
    else:
        outdir = args.outdir
    os.makedirs(outdir, exist_ok=True)

    components = load_trained_components(cfg)
    data_std = components["data_std"]
    scaler = components["scaler"]
    util_model = components["util_model"]
    model_id = components["model_id"]
    blinder_model = components["blinder_model"]
    ptb_mean_per_lead = components["ptb_mean_per_lead"]
    ptb_std_per_lead = components["ptb_std_per_lead"]

    T_target, C_target = data_std.shape[1], data_std.shape[2]

    # ----------------------------------------------------
    # Load + align + PTB-normalize + standardize custom ECG
    # ----------------------------------------------------
    ecg_raw_tc = load_and_align_custom_ecg(
        args.mat_path,
        T_target,
        C_target,
        ptb_mean_per_lead,
        ptb_std_per_lead,
    )  # (T,C) in PTB-like units

    ecg_std_tc = standardize_ecg(ecg_raw_tc, scaler)  # (1,T,C)

    # ----------------------------------------------------
    # Identity + utility on ORIGINAL (PTB-normalized) ECG
    # ----------------------------------------------------
    print("[Eval] Running identity/utility on original PTB-normalized ECG...")
    id_preds_orig = run_identity_on_ecg(model_id, ecg_std_tc, cfg)
    util_probs_orig = run_utility_on_ecg(util_model, ecg_std_tc)

    # ----------------------------------------------------
    # Anonymization
    # ----------------------------------------------------
    print("[Anon] Running Blinder VAE on custom ECG...")
    ecg_std_anon_tc = anonymize_single_ecg(blinder_model, ecg_std_tc, cfg)   # (1,T,C)

    # ----------------------------------------------------
    # Identity + utility on ANONYMIZED
    # ----------------------------------------------------
    print("[Eval] Running identity/utility on anonymized ECG...")
    id_preds_anon = run_identity_on_ecg(model_id, ecg_std_anon_tc, cfg)
    util_probs_anon = run_utility_on_ecg(util_model, ecg_std_anon_tc)

    # ----------------------------------------------------
    # Fidelity metrics + FFT + overlay (in PTB-like raw units)
    # ----------------------------------------------------
    print("[Fidelity] Computing RMSE + PSD corr + plots...")

    ecg_raw_anon_tc = inverse_standardize_ecg(ecg_std_anon_tc, scaler)  # (T,C)

    fidelity = {
        "rmse": rmse(ecg_raw_tc, ecg_raw_anon_tc),
        "psd_corr": psd_correlation(
            ecg_raw_tc, ecg_raw_anon_tc, fs=cfg.sampling_frequency
        ),
    }
    print("  rmse:", fidelity["rmse"])
    print("  psd_corr:", fidelity["psd_corr"])

    # FFT before/after
    fft_raw_path = os.path.join(outdir, "fft_custom_raw.png")
    fft_anon_path = os.path.join(outdir, "fft_custom_anon.png")
    plot_fft(ecg_raw_tc, cfg.sampling_frequency, "Custom ECG FFT (raw, PTB-like)", fft_raw_path)
    plot_fft(
        ecg_raw_anon_tc,
        cfg.sampling_frequency,
        "Custom ECG FFT (Blinder anon, PTB-like)",
        fft_anon_path,
    )

    # Time-domain overlay for a couple of leads
    overlay_dir = os.path.join(outdir, "overlays")
    os.makedirs(overlay_dir, exist_ok=True)
    leads_to_plot = [0, 1]
    for lead in leads_to_plot:
        out_path = os.path.join(overlay_dir, f"overlay_lead{lead}.png")
        title = "Custom ECG (PTB-normalized): Original vs Blinder anonymized"
        plot_overlay_ecg(
            orig=ecg_raw_tc,
            anon=ecg_raw_anon_tc,
            fs=cfg.sampling_frequency,
            title=title,
            filepath=out_path,
            lead=lead,
            max_seconds=5.0,
        )

    # ----------------------------------------------------
    # Save everything to a JSON summary
    # ----------------------------------------------------
    summary = {
        "mat_path": args.mat_path,
        "identity_predictions_original": id_preds_orig,
        "identity_predictions_anonymized": id_preds_anon,
        "utility_probs_original": util_probs_orig.tolist(),
        "utility_probs_anonymized": util_probs_anon.tolist(),
        "fidelity": fidelity,
        "plots": {
            "fft_raw": fft_raw_path,
            "fft_anon": fft_anon_path,
            "overlay_dir": overlay_dir,
        },
    }
    out_json = os.path.join(outdir, "custom_ecg_results.json")
    with open(out_json, "w") as f:
        json.dump(summary, f, indent=2)

    print("=== Done (custom ECG inference) ===")
    print(f"Summary JSON: {out_json}")

In [ ]:
if __name__ == "__main__":
    main()